# 07 — Exposure

Who pays for the emissions in notebook 04.

The ASR says whether a country lives inside its share of the carbon budget.
It says nothing about what happens to that country when the budget is
overshot. This notebook joins a **climate exposure** axis onto the ASR panel
so the two can be read against each other in one scatter.

## Why ND-GAIN and not INFORM or the WorldRiskIndex

Most global risk indices score *absolute expected humanitarian impact*, which
is population-weighted. A country of 11,000 people cannot rank high on them no
matter what happens to it: INFORM Risk 2026 puts Tuvalu 180th of 191 on hazard,
and the WorldRiskIndex 2025 puts it 164th of 193. Both would draw the Pacific as
safe, which is the opposite of the argument here and would be wrong.

ND-GAIN is structural rather than population-weighted, so smallness does not
read as safety.

## Why exposure and not ND-GAIN's headline vulnerability score

ND-GAIN vulnerability is the mean of three components — exposure, sensitivity
and adaptive capacity. The last two are largely development indicators, and it
shows: across the 183 countries with both numbers at 2023, the headline
vulnerability score correlates **-0.83** with log GDP per capita, adaptive
capacity **-0.86**, sensitivity **-0.66**. A chart built on the composite is
open to the fair reply that it plots poverty and calls it climate.

**Exposure** is the physical component alone — projected change in sea level,
temperature, precipitation, cereal yield, marine biodiversity, and the share of
population and infrastructure in the way of it. It correlates **-0.50** with
GDP per capita: still not independent of wealth, because poor countries really
do sit in worse places, but far from a restatement of it. On exposure Tuvalu
ranks 2nd of 192 countries behind the Maldives, and six Pacific islands sit in
the global top 20.

The composite is still in the same download if it is ever wanted — swap the
filename below. The correlations above are reproduced in the last cell.

The SPC Pacific Data Hub has no exposure index of its own — checked across all
127 dataflows. Its nearest equivalent, `DF_POP_LECZ` (share of population in the
low-elevation coastal zone), is Pacific-only and so cannot place the Pacific
against the world on one axis.


In [ ]:
import io
import json
import zipfile

import numpy as np
import pandas as pd
import requests

from config import PACIFIC_ISO3, ROOT, VIZ, YEARS

YEAR = max(YEARS)  # 2023
Y = str(YEAR)

# ND-GAIN publishes the Country Index as one zip of CSVs, updated annually.
# The 2026 edition carries 1995-2024. The asset id changes with each edition:
# check https://gain.nd.edu/our-work/country-index/download-data/ if this 404s.
NDGAIN_URL = "https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip"
NDGAIN_DIR = ROOT / "data_raw" / "ndgain"

# Both live under vulnerability/ in the archive — exposure is a component of
# the composite, not a separate index.
WANTED = ["vulnerability/exposure.csv", "vulnerability/vulnerability.csv"]
EXPOSURE_CSV = NDGAIN_DIR / "exposure.csv"
VULN_CSV = NDGAIN_DIR / "vulnerability.csv"


## 1. Download

The zip is ~4.7 MB and holds every ND-GAIN indicator; we keep two tables. The
server rejects requests without a browser user agent — it answers a bare
request with a 403, not the file.


In [ ]:
if not all((NDGAIN_DIR / f.split("/")[-1]).exists() for f in WANTED):
    NDGAIN_DIR.mkdir(parents=True, exist_ok=True)
    resp = requests.get(
        NDGAIN_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=300
    )
    resp.raise_for_status()
    print(f"downloaded {len(resp.content) / 1e6:.1f} MB")

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        # The archive's top folder has been named both 'resources' and
        # 'resources 2' across editions, so match on the tail of the path.
        for wanted in WANTED:
            members = [
                n for n in zf.namelist()
                if n.endswith(wanted) and not n.startswith("__MACOSX")
            ]
            assert len(members) == 1, f"expected one {wanted}, found {members}"
            (NDGAIN_DIR / wanted.split("/")[-1]).write_bytes(zf.read(members[0]))

print(EXPOSURE_CSV)


## 2. Join

Three tables meet on ISO3, all at one year:

- **ASR** — the three allocation rules from notebook 03, `{iso: {year: ratio}}`
- **ND-GAIN exposure** — wide, one column per year 1995-2024
- **GDP per capita** — World Bank PPP in constant 2017 USD, downloaded by
  pyaesa in notebook 01; the same GDP the prioritarian rule is built on, so
  the alternative x axis and the `pr` series are on one definition.


In [ ]:
countries = pd.read_csv(VIZ / "countries.csv").set_index("iso_code")

asr = {
    "eg": json.load(open(VIZ / "asr.json")),
    "gf": json.load(open(VIZ / "asr_gf.json")),
    "pr": json.load(open(VIZ / "asr_gdp.json")),
}

exposure = pd.read_csv(EXPOSURE_CSV).set_index("ISO3")[Y].dropna()

wb = pd.read_csv(ROOT / "data_processed" / "pop_gdp" / "wb_processed.csv")
wb_gdp = wb[wb["variable"] == "GDP|PPP"].set_index("iso3_code")[Y]
wb_pop = wb[wb["variable"] == "Population"].set_index("iso3_code")[Y]
gdp_pc = (wb_gdp / wb_pop).dropna()

print(f"{len(asr['eg'])} countries with an ASR")
print(f"{len(exposure)} with an ND-GAIN exposure score in {YEAR}")
print(f"{len(gdp_pc)} with GDP per capita in {YEAR}")


In [ ]:
def row(iso):
    """One country. `exposure` and `gdp` are None where the source has no
    entry; the scatter drops a point rather than inventing a position."""
    return {
        "iso": iso,
        "name": countries.loc[iso, "name"],
        "pacific": iso in PACIFIC_ISO3,
        "exposure": round(float(exposure[iso]), 4) if iso in exposure.index else None,
        "gdp": round(float(gdp_pc[iso]), 1) if iso in gdp_pc.index else None,
        "asr": {
            rule: (round(table[iso][Y], 4) if iso in table and Y in table[iso] else None)
            for rule, table in asr.items()
        },
    }


panel = [row(iso) for iso in sorted(asr["eg"])]
print(f"{len(panel)} countries in the panel")


## 3. What the join drops

Neither gap is fixable here and both are the same gap: an index of *countries*
has no row for a territory that is not a country. New Caledonia and French
Polynesia have an ASR — SPC reports their emissions and pyaesa allocates to
them — but no ND-GAIN score and no World Bank GDP, so they carry an ASR and no
x position. This is the same exclusion already noted for American Samoa, Guam
and the Northern Marianas in `config.py`, one step further along.


In [ ]:
plotted = {
    axis: sum(1 for c in panel if c[axis] is not None and c["asr"]["eg"] is not None)
    for axis in ("exposure", "gdp")
}
print(f"plottable against exposure: {plotted['exposure']}")
print(f"plottable against GDP per capita: {plotted['gdp']}")

missing = [c["iso"] for c in panel if c["exposure"] is None]
print(f"\nno exposure score ({len(missing)}): {', '.join(missing)}")

pacific_gaps = [c["iso"] for c in panel if c["pacific"] and c["exposure"] is None]
assert set(pacific_gaps) <= {"NCL", "PYF"}, pacific_gaps
print(f"Pacific gaps: {', '.join(pacific_gaps)} — territories, as expected")


## 4. Where the Pacific lands

The scene's whole claim, as a table: exposure high, ASR at or under a fair
share. World rank is out of the 192 countries ND-GAIN scores, 1 = most exposed.


In [ ]:
world_rank = exposure.rank(ascending=False, method="min").astype(int)

pacific = pd.DataFrame([c for c in panel if c["pacific"]])
pacific["asr_eg"] = pacific["asr"].map(lambda a: a["eg"])
pacific["world_rank"] = pacific["iso"].map(world_rank)

print(
    pacific[["iso", "name", "exposure", "world_rank", "gdp", "asr_eg"]]
    .sort_values("exposure", ascending=False)
    .to_string(index=False)
)


## 5. The reason for exposure over the composite

Reproduces the correlations quoted at the top. Every ND-GAIN aggregate is
measured against log GDP per capita, because the objection this answers is
that a vulnerability axis is a wealth axis in disguise.


In [ ]:
comparison = pd.DataFrame({
    "vulnerability": pd.read_csv(VULN_CSV).set_index("ISO3")[Y],
    "exposure": exposure,
    "gdp": gdp_pc,
}).dropna()

log_gdp = np.log10(comparison["gdp"])
print(f"across {len(comparison)} countries with both numbers at {YEAR}:")
for column in ("vulnerability", "exposure"):
    print(f"  {column:14s} r vs log GDP per capita = {comparison[column].corr(log_gdp):+.2f}")


## 6. Export

`scatter.json` is the fifth table the app fetches at runtime. The year is
carried in the file so the scene can label itself without hard-coding 2023 in
two places.


In [ ]:
out = {
    "year": YEAR,
    "source": {
        "exposure": "ND-GAIN Country Index 2026, vulnerability/exposure (University of Notre Dame)",
        "gdp": "World Bank GDP per capita, PPP, constant 2017 USD",
        "asr": "pyaesa, notebook 03",
    },
    "countries": panel,
}

path = VIZ / "scatter.json"
path.write_text(json.dumps(out, separators=(",", ":")))
print(f"wrote {path} — {path.stat().st_size / 1024:.0f} KB")
